# 工具的应用案例

## 1、使用args_schema

举例：arg_schema给出明确的参数信息

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools.convert import tool
from rich import print as rprint

class WeatherSchema(BaseModel):
    city : str = Field(
        description="具体的城市"
    )
    if_forecast : bool = Field(
        default=True,
        description="是否包含明日天气"
    )

# 自定义工具
@tool(name_or_callable="get_weather_and_forecast", description="查询当日天气，可以包含明天的天气预报",        args_schema=WeatherSchema)
def get_weather(city : str, if_forecast : bool):
    """"""
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当日天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '具体的城市', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [
    HumanMessage("今天仲恺的天气怎么样？明天呢？")
]

# 3、调用模型 -> AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所有此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型 -> AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for message in messages:
    message.pretty_print()

================================ Human Message =================================

今天仲恺的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_6b4824373cfb4502a968d7a8)
 Call ID: call_6b4824373cfb4502a968d7a8
  Args:
    city: 仲恺
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

仲恺今天天气不错
明天下雨
================================== Ai Message ==================================

今天仲恺的天气不错，不过明天会下雨，出门记得带好雨具哦！


## 2、撰写docstring
可以在docstring中撰写参数的描述信息，此时参数默认值和类型都要通过函数签名传递。

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool
# from pydantic import BaseModel, Field
from langchain_core.tools.convert import tool
from rich import print as rprint



# 自定义工具
@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city : str = "beijing", if_forecast : bool = False):
    """
    查询当日的天气，可以包含明天的天气预报

    Args:
        city : 城市名称
        if_forecast : 是否包含明天天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当日的天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'default': 'beijing', 'description': '城市名称', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明天天气', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [11]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [
    HumanMessage("今天仲恺的天气怎么样？明天呢？")
]

# 3、调用模型 -> AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所有此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型 -> AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for message in messages:
    message.pretty_print()

================================ Human Message =================================

今天仲恺的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_f3b09a22381446e69e3863b8)
 Call ID: call_f3b09a22381446e69e3863b8
  Args:
    city: 仲恺
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

仲恺今天天气不错
明天下雨
================================== Ai Message ==================================

今天仲恺的天气挺不错的，不过明天会下雨，出门记得带伞哦！
